# Credit Risk: Boruta Then VIF

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "pyproject.toml").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from catboost_utility.boruta_catboost import BorutaCatBoost
from catboost_utility.vif_catboost import CatBoostVIF

EXAMPLES_ROOT = PROJECT_ROOT / "examples"
CATBOOST_EXPLORATION_PARAMS = {"iterations": 50, "depth": 4, "learning_rate": 0.1}

In [2]:
data_path = EXAMPLES_ROOT / "credit_risk_data" / "credit_risk.csv"
df = pd.read_csv(data_path)

print(f"Loaded {data_path.name} with shape {df.shape}")
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

Loaded credit_risk.csv with shape (1000, 21)


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,... < 100 DM,6,critical account/other credits existing,domestic appliances,1169,unknown/no savings account,... >= 7 years,4,male : single,none,...,real estate,67,none,own,2,skilled employee/official,1,yes,yes,1
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,none,...,real estate,22,none,own,1,skilled employee/official,1,no,yes,0
2,no checking account,12,critical account/other credits existing,retraining,2096,... < 100 DM,4 <= ... < 7 years,2,male : single,none,...,real estate,49,none,own,1,unskilled - resident,2,no,yes,1
3,... < 100 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,guarantor,...,building society savings agreement/life insurance,45,none,for free,1,skilled employee/official,2,no,yes,1
4,... < 100 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,none,...,unknown/no property,53,none,for free,2,skilled employee/official,2,no,yes,0


,dtype
status,str
duration,int64
credit_history,str
purpose,str
amount,int64
savings,str
employment_duration,str
installment_rate,int64
personal_status_sex,str
other_debtors,str


In [3]:
target = "credit_risk"
X = df.drop(columns=[target]).copy()
y = df[target].astype(int)

for col in X.select_dtypes(include=["object", "category", "bool"]).columns:
    X[col] = X[col].fillna("missing")

cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print(f"Prepared X with shape {X.shape} and target '{target}'")
print("Categorical features:", cat_features)

Prepared X with shape (1000, 20) and target 'credit_risk'
Categorical features: ['status', 'credit_history', 'purpose', 'savings', 'employment_duration', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21164\1035596137.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category", "bool"]).columns:
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21164\1035596137.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pyda

In [4]:
boruta = BorutaCatBoost(
    cat_features=cat_features,
    max_iter=15,
    patience=3,
    correction_method="bonferroni",
    task_type="classification",
    catboost_params=CATBOOST_EXPLORATION_PARAMS,
    random_state=42,
)
boruta.fit(X, y)

boruta_selected = boruta.get_feature_names_out()
boruta_result = boruta.get_selection_result()
decision_log = boruta.decision_log_.copy()
if decision_log.empty:
    latest_boruta_decisions = decision_log
else:
    status_order = pd.CategoricalDtype(["confirmed", "tentative", "rejected"], ordered=True)
    latest_boruta_decisions = (
        decision_log.assign(status=decision_log["status"].astype(status_order))
        .sort_values(["feature", "iteration"])
        .groupby("feature", group_keys=False)
        .tail(1)
        .sort_values(["status", "feature"])
        .reset_index(drop=True)
    )

print("Boruta selected features:", boruta_selected)
print("Boruta rejected features:", boruta_result.rejected_features)
print("Boruta tentative features:", boruta_result.tentative_features)
display(latest_boruta_decisions)

if not boruta_selected:
    raise RuntimeError(
        "Boruta did not confirm any features. Increase max_iter or CatBoost iterations and rerun."
    )

Boruta selected features: ['status', 'duration', 'credit_history', 'amount', 'savings']
Boruta rejected features: ['employment_duration', 'personal_status_sex', 'other_debtors', 'present_residence', 'housing', 'number_credits', 'job', 'people_liable', 'telephone', 'foreign_worker']
Boruta tentative features: ['purpose', 'installment_rate', 'property', 'age', 'other_installment_plans']


,iteration,iteration_seed,feature,shadow_max,hits,p_upper,p_lower,adj_p_upper,adj_p_lower,status
0,9,51,amount,2.203252,9,0.001953,1.000000,0.039062,1.000000,confirmed
1,9,51,credit_history,2.203252,9,0.001953,1.000000,0.039062,1.000000,confirmed
2,9,51,duration,2.203252,9,0.001953,1.000000,0.039062,1.000000,confirmed
3,9,51,savings,2.203252,9,0.001953,1.000000,0.039062,1.000000,confirmed
4,9,51,status,2.203252,9,0.001953,1.000000,0.039062,1.000000,confirmed
5,15,57,age,3.781531,9,0.303619,0.849121,1.000000,1.000000,tentative
6,15,57,installment_rate,3.781531,6,0.849121,0.303619,1.000000,1.000000,tentative
7,15,57,other_installment_plans,3.781531,6,0.849121,0.303619,1.000000,1.000000,tentative
8,15,57,property,3.781531,3,0.996307,0.017578,1.000000,0.087891,tentative
9,15,57,purpose,3.781531,7,0.696381,0.500000,1.000000,1.000000,tentative


In [5]:
X_boruta = X[boruta_selected].copy()
vif_cat_features = X_boruta.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

vif = CatBoostVIF(
    cat_features=vif_cat_features,
    threshold=5.0,
    scoring_method="holdout",
    holdout_fraction=0.2,
    n_jobs=1,
    catboost_params=CATBOOST_EXPLORATION_PARAMS,
    random_state=42,
)
vif_result = vif.fit_eliminate(X_boruta)

print("VIF-retained features:", vif_result.selected_features)
print("VIF-dropped features:", vif_result.rejected_features)
display(vif_result.metrics)

elimination_history = pd.DataFrame(vif_result.config["elimination_history"])
if elimination_history.empty:
    print("No VIF eliminations were needed at the current threshold.")
else:
    display(elimination_history)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21164\1810042448.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  vif_cat_features = X_boruta.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


VIF-retained features: ['duration', 'amount', 'status', 'credit_history', 'savings']
VIF-dropped features: []


,feature,vif,r_squared,is_categorical,clamped
0,duration,1.689992,0.408281,False,False
1,amount,1.654748,0.395678,False,False
2,status,1.000000,0.000000,True,True
3,credit_history,1.000000,0.000000,True,True
4,savings,1.000000,0.000000,True,True


No VIF eliminations were needed at the current threshold.


In [6]:
from catboost_utility.rfe_catboost import CatBoostRFE

rfe = CatBoostRFE(
    n_features_to_select=5,
    cat_features=vif_cat_features,
    task_type="classification",
    random_state=42,
)
rfe.fit(X_boruta, y)
rfe_selected = rfe.get_feature_names_out()
print("RFE selected features:", rfe_selected)

X_final = X_boruta[rfe_selected].copy()

print("Final feature set:", rfe_selected)
display(X_final.head())

Final feature set: ['duration', 'amount', 'status', 'credit_history', 'savings']


,duration,amount,status,credit_history,savings
0,6,1169,... < 100 DM,critical account/other credits existing,unknown/no savings account
1,48,5951,0 <= ... < 200 DM,existing credits paid back duly till now,... < 100 DM
2,12,2096,no checking account,critical account/other credits existing,... < 100 DM
3,42,7882,... < 100 DM,existing credits paid back duly till now,... < 100 DM
4,24,4870,... < 100 DM,delay in paying off in the past,... < 100 DM
